In [1]:
import os, tempfile, subprocess

In [2]:
os.environ["BIOEMU_COLABFOLD_DIR"] = "bioemu_colabfold"
os.environ["HPACKER_ENV_NAME"] = "bioemu_hpacker"

In [3]:
seq = "GSSHHHHHHSSGLVPRGSHMASLTEIEHLVQSVCKSYRETCQLRLEDLLRQRSNIFSREEVTGYQRKSMWEMWERCAHHLTEAIQYVVEFAKRLSGFMELCQNDQIVLLKAGAMEVVLVRMCRAYNADNRTVFFEGKYGGMELFRALGCSELISSIFDFSHSLSALHFSEDEIALYTALVLINAHRPGLQEKRKVEQLQYNLELAFHHHLHKTHRQSILAKLPPKGKLRSLCSQHVERLQIFQHLHPIVVQAAFPPLYKELFS"

In [4]:
outf = "test.a3m"

In [5]:
outdir = 'bioemu_test_150_batch75'

# Calculate

In [ ]:
if not os.path.isfile(outf):
    mmseqs = "/home/fnerin/miniconda3/envs/mmseqs/bin/mmseqs"
    db = "/data/fnerin/uniref30_2302/uniref30_2302_db"
    
    with tempfile.TemporaryDirectory() as tmpdir:    
        fastaf = f"{tmpdir}/fasta.fasta"
        with open(fastaf, "w") as fasta:
            fasta.write(f">fasta\n{seq}\n") 
    
        subprocess.run(
            (
                f"{mmseqs} createdb {fastaf} {tmpdir}/qdb -v 1 && "
                + f"{mmseqs} search {tmpdir}/qdb {db} {tmpdir}/result {tmpdir}/tmp  --gpu 1 --num-iterations 3 -e 0.1 --max-seqs 20000 --prefilter-mode 1 -a -v 1 && " # -s 9 ignored with GPU
                + f"{mmseqs} mvdb {tmpdir}/tmp/latest/profile_1 {tmpdir}/prof_res && "
                + f"{mmseqs} lndb {tmpdir}/qdb_h {tmpdir}/prof_res_h && "
                + f"{mmseqs} align {tmpdir}/prof_res {db} {tmpdir}/result {tmpdir}/result_realign -e 10 --max-accept 1000000 --alt-ali 10 -a -v 1 && "
                + f"{mmseqs} filterresult {tmpdir}/qdb {db} {tmpdir}/result_realign {tmpdir}/result_filter --qid 0 --qsc 0.8 --diff 0 --max-seq-id 1.0 --filter-min-enable 100 -v 1 && "
                + f"{mmseqs} result2msa {tmpdir}/qdb {db} {tmpdir}/result_filter {tmpdir}/result_msa --msa-format-mode 6 --filter-msa 1 --filter-min-enable 1000 --diff 3000 --qid 0.0,0.2,0.4,0.6,0.8,1.0 --qsc 0 --max-seq-id 0.95 -v 1 && "
                + f"cp {tmpdir}/result_msa {outf}"
            ),
            shell=True, check=True
        )

rmdb /tmp/tmptwyaggea/tmp/4224541976127059445/pref_tmp_1 

Time for processing: 0h 0m 0s 0ms
mergedbs /tmp/tmptwyaggea/tmp/4224541976127059445/profile_0 /tmp/tmptwyaggea/tmp/4224541976127059445/aln_1 /tmp/tmptwyaggea/tmp/4224541976127059445/aln_0 /tmp/tmptwyaggea/tmp/4224541976127059445/aln_tmp_1 

Merging the results to /tmp/tmptwyaggea/tmp/4224541976127059445/aln_1
[=================================================================] 1 0s 0ms
Time for merging to aln_1: 0h 0m 0s 0ms
Time for processing: 0h 0m 0s 0ms
rmdb /tmp/tmptwyaggea/tmp/4224541976127059445/aln_0 

Time for processing: 0h 0m 0s 0ms
rmdb /tmp/tmptwyaggea/tmp/4224541976127059445/aln_tmp_1 

Time for processing: 0h 0m 0s 0ms
rmdb /tmp/tmptwyaggea/tmp/4224541976127059445/pref_tmp_2 

Time for processing: 0h 0m 0s 0ms
mergedbs /tmp/tmptwyaggea/tmp/4224541976127059445/profile_1 /tmp/tmptwyaggea/result /tmp/tmptwyaggea/tmp/4224541976127059445/aln_1 /tmp/tmptwyaggea/tmp/4224541976127059445/aln_tmp_2 

Merging the results to

In [5]:
from bioemu.sample import main as sample

In [ ]:
%%time
sample(sequence=outf, num_samples=150, batch_size_100 = 75, output_dir=outdir)

Sampling batches...:  67%|█████████████████████████████████████████▎                    | 10/15 [02:32<01:16, 15.30s/it]

In [15]:
os.makedirs(f"{outdir}/pdbs")

In [12]:
import mdtraj
import numpy as np

In [13]:
traj = mdtraj.load(f"{outdir}/samples.xtc", top=f"{outdir}/topology.pdb")
sample_indices = np.arange(traj.n_frames)
for idx in sample_indices:
    traj[idx].save_pdb(f"{outdir}/pdbs/sample_{idx}.pdb")

# Cluster

## foldseek

In [14]:
!../training_data/utils/external/foldseek easy-cluster -h

usage: foldseek easy-cluster <i:PDB|mmCIF[.gz]> ... <i:PDB|mmCIF[.gz]> <o:clusterPrefix> <tmpDir> [options]
 By Martin Steinegger <martin.steinegger@snu.ac.kr>
options: prefilter:                      
 --seed-sub-mat TWIN              Substitution matrix file for k-mer generation [aa:3di.out,nucl:3di.out]
 -s FLOAT                         Sensitivity: 1.0 faster; 4.0 fast; 7.5 sensitive [4.000]
 -k INT                           k-mer length (0: automatically set to optimum) [0]
 --target-search-mode INT         target search mode (0: regular k-mer, 1: similar k-mer) [0]
 --k-score TWIN                   k-mer threshold for generating similar k-mer lists [seq:2147483647,prof:2147483647]
 --max-seqs INT                   Maximum results per query sequence allowed to pass the prefilter (affects sensitivity) [300]
 --split INT                      Split input into N equally distributed chunks. 0: set the best split automatically [0]
 --split-mode INT                 0: split target db; 1:

In [29]:
tmscore_threshold = 0.85
coverage_threshold = 0.9
seq_id = 0.9
coverage_mode = 1

with tempfile.TemporaryDirectory() as temp_dir:
    res = subprocess.run(
        " ../training_data/utils/external/foldseek easy-cluster "
        + f" {outdir}/pdbs "
        + f" {outdir}/cluster "
        + temp_dir
        + f" -c {coverage_threshold} "
        + f" --min-seq-id {seq_id} "
        + f" --tmscore-threshold {tmscore_threshold} "
        + f" --cov-mode {coverage_mode} "
        + f" --single-step-clustering ",
        shell=True,
    )
    assert res.returncode == 0, "Something went wrong with foldseek"

easy-cluster bioemu_test_150_batch75/pdbs bioemu_test_150_batch75/cluster /tmp/tmpulo5mtvh -c 0.9 --min-seq-id 0.9 --tmscore-threshold 0.85 --cov-mode 1 --single-step-clustering 

MMseqs Version:                     	941cd33ff0771cd2e3f144e3293e22a2b87e9fda
Substitution matrix                 	aa:3di.out,nucl:3di.out
Seed substitution matrix            	aa:3di.out,nucl:3di.out
Sensitivity                         	4
k-mer length                        	0
Target search mode                  	0
k-score                             	seq:2147483647,prof:2147483647
Max sequence length                 	65535
Max results per query               	300
Split database                      	0
Split mode                          	2
Split memory limit                  	0
Coverage threshold                  	0.9
Coverage mode                       	1
Compositional bias                  	1
Compositional bias                  	1
Diagonal scoring                    	true
Exact k-mer matching              

In [2]:
import pandas as pd

INFO:numexpr.utils:Note: NumExpr detected 32 cores but "NUMEXPR_MAX_THREADS" not set, so enforcing safe limit of 16.
INFO:numexpr.utils:NumExpr defaulting to 16 threads.


In [10]:
clusters = pd.read_csv(f"{outdir}/cluster_cluster.tsv", sep="\t", header=None)
clusters.value_counts(0)

0
sample_0      111
sample_104      5
sample_13       2
sample_4        2
sample_5        1
sample_50       1
sample_60       1
sample_87       1
Name: count, dtype: int64

In [13]:
import mdtraj as md

frames = [md.load_pdb(f"{outdir}/pdbs/{f}.pdb") for f in clusters[0].unique()]
traj = md.join(frames)
traj[0].save(f"{outdir}/traj.pdb")
traj.save_xtc(f"{outdir}/traj.xtc")

## EnsembleFlex

https://github.com/chembl/EnsembleFlex

Meh

## BioBB

https://github.com/bioexcel/biobb_wf_flexdyn/blob/main/biobb_wf_flexdyn/notebooks/biobb_wf_flexdyn.ipynb

gromacs

## TTClust

In [32]:
os.makedirs("ttclust")

In [35]:
subprocess.run(
    "ttclust "
    + f" -f {outdir}/samples.xtc"
    + f" -t {outdir}/topology.pdb "
    + f" -l ttclust/log.log ",
    shell=True,
)

|>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>| Time:  0:00:00 |<<<<<<<<<<<<<<<<<<<<<<<<<<<<<|
/home/fnerin/miniconda3/envs/bioemu/lib/python3.12/site-packages/ttclust/ttclust.py:726: ClusterWarning: The symmetric non-negative hollow observation matrix looks suspiciously like an uncondensed distance matrix
  linkage = sch.linkage(distances, method=args["method"])
/home/fnerin/miniconda3/envs/bioemu/lib/python3.12/site-packages/sklearn/manifold/_mds.py:677: FutureWarning: The default value of `n_init` will change from 4 to 1 in 1.9.
  warnings.warn(
<mdtraj.Trajectory with 124 frames, 1303 atoms, 263 residues, without unitcells>


********************************************************
******************  TTCLUST 4.10.1 *********************
********************************************************

======= TRAJECTORY READING =======
====== Clustering ========
         creating distance matrix
NOTE : Extraction of subtrajectory for time optimisation
Calculation ended - saving distance matrix
Saving distance matrix : backbone.npy
         Matrix shape: (124, 124)
         Scipy linkage in progress. Please wait. It can be long
         >Done!
         ...Saving linkage matrix...
Saving linkage matrix : backbone--backbone_linkage_ward.npy
         >Done!
  cutoff for clustering : 8.31

**** Cluster Results
====== Reordering clusters ======
====== Generating Graph ======
====== Calc. repr. frame  ======
Searching for representative frames
cluster 1
    size = 30
    representative frame=59
    spread  : 5.42 
    Frames : [1, 7, 9, 14, 17, 19, 20, 22, 24, 28, 36, 38, 44, 49, 58, 60, 61, 67, 70, 73, 79, 83, 86, 90, 9

CompletedProcess(args='ttclust  -f bioemu_test_150_batch75/samples.xtc -t bioemu_test_150_batch75/topology.pdb  -l ttclust/log.log ', returncode=1)

## protein-cluster-conformers

**Incompatible with needed new numpy**

# foldseek Side-chains

In [ ]:
from bioemu import sidechain_relax

## Only sc

In [26]:
outsc = f"{outdir}/relax_nomd"
os.makedirs(outsc, exist_ok=True)

In [ ]:
%%time
sidechain_relax.main(
    pdb_path=f"{outdir}/traj.pdb", 
    xtc_path=f"{outdir}/traj.xtc",
    outpath=outsc, 
    prefix="out",
    md_equil=False,
    # md_protocol=md_protocol,
)

reconstructing side-chains:   0%|          | 0/8 [00:00<?, ?it/s]

CPU times: user 366 ms, sys: 27.1 ms, total: 393 ms
Wall time: 2min 16s


In [28]:
for i, frame in enumerate(mdtraj.load(f"{outsc}/out_sidechain_rec.xtc", top=f"{outsc}/out_sidechain_rec.pdb")):
    frame.save_pdb(f"{outsc}/frame_{i}.pdb")

## sc + minim

In [ ]:
outsc = f"{outdir}/relax_minim"
os.makedirs(outsc)

In [7]:
%%time
sidechain_relax.main(
    pdb_path=f"{outdir}/traj.pdb", 
    xtc_path=f"{outdir}/traj.xtc",
    outpath=outsc, 
    prefix="out",
    md_equil=True,
    md_protocol=sidechain_relax.MDProtocol.LOCAL_MINIMIZATION,
)

reconstructing side-chains:   0%|          | 0/8 [00:00<?, ?it/s]

running MD equilibration:   0%|          | 0/8 [00:00<?, ?it/s]

DEBUG:bioemu.sidechain_relax:creating MD setup
DEBUG:bioemu.md_utils:adding constraint force with k=1000
DEBUG:bioemu.sidechain_relax:simulation uses CUDA platform
DEBUG:bioemu.sidechain_relax:running local energy minimization
DEBUG:bioemu.sidechain_relax:creating MD setup
DEBUG:bioemu.md_utils:adding constraint force with k=1000
DEBUG:bioemu.sidechain_relax:simulation uses CUDA platform
DEBUG:bioemu.sidechain_relax:running local energy minimization
DEBUG:bioemu.sidechain_relax:creating MD setup
DEBUG:bioemu.md_utils:adding constraint force with k=1000
DEBUG:bioemu.sidechain_relax:simulation uses CUDA platform
DEBUG:bioemu.sidechain_relax:running local energy minimization
DEBUG:bioemu.sidechain_relax:creating MD setup
DEBUG:bioemu.md_utils:adding constraint force with k=1000
DEBUG:bioemu.sidechain_relax:simulation uses CUDA platform
DEBUG:bioemu.sidechain_relax:running local energy minimization
DEBUG:bioemu.sidechain_relax:creating MD setup
DEBUG:bioemu.md_utils:adding constraint force

CPU times: user 4min 58s, sys: 3.11 s, total: 5min 1s
Wall time: 8min 58s


In [ ]:
for i, frame in enumerate(mdtraj.load(f"{outsc}/out_sidechain_rec.xtc", top=f"{outsc}/out_sidechain_rec.pdb")):
    frame.save_pdb(f"{outsc}/frame_{i}.pdb")

## Parallel minim

In [6]:
import mdtraj

In [7]:
traj = mdtraj.load_xtc(f"{outdir}/relax_nomd/out_sidechain_rec.xtc", top=f"{outdir}/relax_nomd/out_sidechain_rec.pdb")

In [8]:
from tqdm.contrib.concurrent import process_map
from tqdm.notebook import tqdm

In [9]:
from bioemu.sidechain_relax import run_one_md

In [30]:
outsc = f"{outdir}/relax_minim_parallel"
os.makedirs(outsc, exist_ok=True)

In [14]:
%%time

def launch_md(kwargs):
    return run_one_md(**kwargs)

frames = process_map(
    launch_md,
    tuple(
        dict(
            frame = frame,
            only_energy_minimization = True,
            simtime_ns = 0.0, # default
            # simtime_ns_nvt_equil: float = 0.1,
            # simtime_ns_npt_equil: float = 0.4,
            outpath = outsc,
            file_prefix = f'frame_{n}',
        )
        for n, frame in enumerate(traj)
    ),
    tqdm_class=tqdm,
    max_workers=8
)

newtraj = mdtraj.join(frames)
newtraj[0].save(f"{outsc}/out_md_equil.pdb")
newtraj.save_xtc(f"{outsc}/out_md_equil.xtc")

  0%|          | 0/8 [00:00<?, ?it/s]

DEBUG:bioemu.sidechain_relax:creating MD setup
DEBUG:bioemu.sidechain_relax:creating MD setup
DEBUG:bioemu.sidechain_relax:creating MD setup
DEBUG:bioemu.sidechain_relax:creating MD setup
DEBUG:bioemu.sidechain_relax:creating MD setup
DEBUG:bioemu.sidechain_relax:creating MD setup
DEBUG:bioemu.sidechain_relax:creating MD setup
DEBUG:bioemu.sidechain_relax:creating MD setup
DEBUG:bioemu.md_utils:adding constraint force with k=1000
DEBUG:bioemu.sidechain_relax:simulation uses CUDA platform
DEBUG:bioemu.md_utils:adding constraint force with k=1000
DEBUG:bioemu.sidechain_relax:simulation uses CUDA platform
DEBUG:bioemu.md_utils:adding constraint force with k=1000
DEBUG:bioemu.md_utils:adding constraint force with k=1000
DEBUG:bioemu.sidechain_relax:simulation uses CUDA platform
DEBUG:bioemu.sidechain_relax:simulation uses CUDA platform
DEBUG:bioemu.md_utils:adding constraint force with k=1000
DEBUG:bioemu.md_utils:adding constraint force with k=1000
DEBUG:bioemu.sidechain_relax:simulation 

CPU times: user 363 ms, sys: 74.7 ms, total: 438 ms
Wall time: 2min 5s


In [31]:
for i, frame in enumerate(mdtraj.load(f"{outsc}/out_md_equil.xtc", top=f"{outsc}/out_md_equil.pdb")):
    frame.save_pdb(f"{outsc}/frame_{i}.pdb")

END


## Parallel equil

In [15]:
import mdtraj

In [16]:
traj = mdtraj.load_xtc(f"{outdir}/relax_nomd/out_sidechain_rec.xtc", top=f"{outdir}/relax_nomd/out_sidechain_rec.pdb")

In [17]:
from tqdm.contrib.concurrent import process_map
from tqdm.notebook import tqdm

In [18]:
from bioemu.sidechain_relax import run_one_md

In [33]:
outsc = f"{outdir}/relax_equil_parallel"
os.makedirs(outsc, exist_ok=True)

In [20]:
%%time

def launch_md(kwargs):
    return run_one_md(**kwargs)

frames = process_map(
    launch_md,
    tuple(
        dict(
            frame = frame,
            only_energy_minimization = False,
            simtime_ns = 0.0, # default
            # simtime_ns_nvt_equil: float = 0.1, # left defaults
            # simtime_ns_npt_equil: float = 0.4, # left defaults
            outpath = outsc,
            file_prefix = f'frame_{n}',
        )
        for n, frame in enumerate(traj)
    ),
    tqdm_class=tqdm,
    max_workers=4
)

newtraj = mdtraj.join(frames)
newtraj[0].save(f"{outsc}/out_md_equil.pdb")
newtraj.save_xtc(f"{outsc}/out_md_equil.xtc")

  0%|          | 0/8 [00:00<?, ?it/s]

DEBUG:bioemu.sidechain_relax:creating MD setup
DEBUG:bioemu.sidechain_relax:creating MD setup
DEBUG:bioemu.sidechain_relax:creating MD setup
DEBUG:bioemu.sidechain_relax:creating MD setup
DEBUG:bioemu.md_utils:adding constraint force with k=1000
DEBUG:bioemu.md_utils:adding constraint force with k=1000
DEBUG:bioemu.sidechain_relax:simulation uses CUDA platform
DEBUG:bioemu.sidechain_relax:simulation uses CUDA platform
DEBUG:bioemu.md_utils:adding constraint force with k=1000
DEBUG:bioemu.sidechain_relax:simulation uses CUDA platform
DEBUG:bioemu.md_utils:adding constraint force with k=1000
DEBUG:bioemu.sidechain_relax:simulation uses CUDA platform
DEBUG:bioemu.sidechain_relax:running local energy minimization
DEBUG:bioemu.sidechain_relax:running local energy minimization


small timestep pre-equilibration:   0%|          | 0/4 [00:00<?, ?it/s]

DEBUG:bioemu.md_utils:running with init integration step of 1e-06 ps


small timestep pre-equilibration:   0%|          | 0/4 [00:00<?, ?it/s]

...

DEBUG:bioemu.md_utils:running with init integration step of 1e-06 ps
DEBUG:bioemu.md_utils:running with init integration step of 1e-05 ps
DEBUG:bioemu.md_utils:running with init integration step of 0.0001 ps
DEBUG:bioemu.md_utils:running with init integration step of 0.001 ps
DEBUG:bioemu.md_utils:running 0.1 ns constrained MD equilibration (NVT)


NVT equilibration (0.1 ns):   0%|          | 0/100 [00:00<?, ?it/s]

DEBUG:bioemu.md_utils:running 0.4 ns constrained MD equilibration (NPT)


NPT equilibration (0.4 ns):   0%|          | 0/100 [00:00<?, ?it/s]

CPU times: user 1.62 s, sys: 873 ms, total: 2.49 s
Wall time: 1h 1min 22s


In [34]:
for i, frame in enumerate(mdtraj.load(f"{outsc}/out_md_equil.xtc", top=f"{outsc}/out_md_equil.pdb")):
    frame.save_pdb(f"{outsc}/frame_{i}.pdb")

## Non-parallel equil

In [23]:
outsc = f"{outdir}/relax_equil_nonparallel"
os.makedirs(outsc)

In [24]:
%%time

frame = run_one_md(**dict(
    frame = traj[0],
    only_energy_minimization = False,
    simtime_ns = 0.0, # default
    # simtime_ns_nvt_equil: float = 0.1, # left defaults
    # simtime_ns_npt_equil: float = 0.4, # left defaults
    outpath = outsc,
    file_prefix = f'non-parallel-equil',
))

DEBUG:bioemu.sidechain_relax:creating MD setup
DEBUG:bioemu.md_utils:adding constraint force with k=1000
DEBUG:bioemu.sidechain_relax:simulation uses CUDA platform
DEBUG:bioemu.sidechain_relax:running local energy minimization


small timestep pre-equilibration:   0%|          | 0/4 [00:00<?, ?it/s]

DEBUG:bioemu.md_utils:running with init integration step of 1e-06 ps
DEBUG:bioemu.md_utils:running with init integration step of 1e-05 ps
DEBUG:bioemu.md_utils:running with init integration step of 0.0001 ps
DEBUG:bioemu.md_utils:running with init integration step of 0.001 ps
DEBUG:bioemu.md_utils:running 0.1 ns constrained MD equilibration (NVT)


NVT equilibration (0.1 ns):   0%|          | 0/100 [00:00<?, ?it/s]

DEBUG:bioemu.md_utils:running 0.4 ns constrained MD equilibration (NPT)


NPT equilibration (0.4 ns):   0%|          | 0/100 [00:00<?, ?it/s]

CPU times: user 8min 31s, sys: 2.91 s, total: 8min 34s
Wall time: 12min 15s


# ttclust Side-chains

In [6]:
import mdtraj as md

frames = [md.load_pdb(f"{outdir}/ttclust/log/{f}") for f in os.listdir(f"{outdir}/ttclust/log") if f.endswith(".pdb")]
traj = md.join(frames)
traj[0].save(f"{outdir}/ttclust/traj.pdb")
traj.save_xtc(f"{outdir}/ttclust/traj.xtc")

In [7]:
from bioemu import sidechain_relax

## Only sc

In [8]:
outsc = f"{outdir}/ttclust_relax_nomd"
os.makedirs(outsc, exist_ok=True)

In [9]:
%%time
sidechain_relax.main(
    pdb_path=f"{outdir}/ttclust/traj.pdb", 
    xtc_path=f"{outdir}/ttclust/traj.xtc",
    outpath=outsc, 
    prefix="out",
    md_equil=False,
    # md_protocol=md_protocol,
)

reconstructing side-chains:   0%|          | 0/4 [00:00<?, ?it/s]

e3nn version:  0.5.0
CPU times: user 141 ms, sys: 12.1 ms, total: 153 ms
Wall time: 34.1 s


In [11]:
for i, frame in enumerate(md.load(f"{outsc}/out_sidechain_rec.xtc", top=f"{outsc}/out_sidechain_rec.pdb")):
    frame.save_pdb(f"{outsc}/frame_{i}.pdb")

## Parallel minim

In [12]:
import mdtraj

In [13]:
traj = mdtraj.load_xtc(f"{outsc}/out_sidechain_rec.xtc", top=f"{outsc}/out_sidechain_rec.pdb")

In [14]:
from tqdm.contrib.concurrent import process_map
from tqdm.notebook import tqdm

In [15]:
from bioemu.sidechain_relax import run_one_md

In [16]:
outsc = f"{outdir}/ttclust_relax_minim_parallel"
os.makedirs(outsc, exist_ok=True)

In [17]:
%%time

def launch_md(kwargs):
    return run_one_md(**kwargs)

frames = process_map(
    launch_md,
    tuple(
        dict(
            frame = frame,
            only_energy_minimization = True,
            simtime_ns = 0.0, # default
            # simtime_ns_nvt_equil: float = 0.1,
            # simtime_ns_npt_equil: float = 0.4,
            outpath = outsc,
            file_prefix = f'frame_{n}',
        )
        for n, frame in enumerate(traj)
    ),
    tqdm_class=tqdm,
    max_workers=4
)

newtraj = mdtraj.join(frames)
newtraj[0].save(f"{outsc}/out_md_equil.pdb")
newtraj.save_xtc(f"{outsc}/out_md_equil.xtc")

  0%|          | 0/4 [00:00<?, ?it/s]

DEBUG:bioemu.sidechain_relax:creating MD setup
DEBUG:bioemu.sidechain_relax:creating MD setup
DEBUG:bioemu.sidechain_relax:creating MD setup
DEBUG:bioemu.sidechain_relax:creating MD setup
DEBUG:bioemu.md_utils:adding constraint force with k=1000
DEBUG:bioemu.sidechain_relax:simulation uses CUDA platform
DEBUG:bioemu.md_utils:adding constraint force with k=1000
DEBUG:bioemu.sidechain_relax:simulation uses CUDA platform
DEBUG:bioemu.md_utils:adding constraint force with k=1000
DEBUG:bioemu.sidechain_relax:simulation uses CUDA platform
DEBUG:bioemu.sidechain_relax:running local energy minimization
DEBUG:bioemu.sidechain_relax:running local energy minimization
DEBUG:bioemu.md_utils:adding constraint force with k=1000
DEBUG:bioemu.sidechain_relax:simulation uses CUDA platform
DEBUG:bioemu.sidechain_relax:running local energy minimization
DEBUG:bioemu.sidechain_relax:running local energy minimization


CPU times: user 179 ms, sys: 54.9 ms, total: 234 ms
Wall time: 4min


In [18]:
for i, frame in enumerate(mdtraj.load(f"{outsc}/out_md_equil.xtc", top=f"{outsc}/out_md_equil.pdb")):
    frame.save_pdb(f"{outsc}/frame_{i}.pdb")

## Parallel equil

In [19]:
outsc = f"{outdir}/ttclust_relax_equil_parallel"
os.makedirs(outsc, exist_ok=True)

In [20]:
%%time

def launch_md(kwargs):
    return run_one_md(**kwargs)

frames = process_map(
    launch_md,
    tuple(
        dict(
            frame = frame,
            only_energy_minimization = False,
            simtime_ns = 0.0, # default
            # simtime_ns_nvt_equil: float = 0.1, # left defaults
            # simtime_ns_npt_equil: float = 0.4, # left defaults
            outpath = outsc,
            file_prefix = f'frame_{n}',
        )
        for n, frame in enumerate(traj)
    ),
    tqdm_class=tqdm,
    max_workers=1
)

newtraj = mdtraj.join(frames)
newtraj[0].save(f"{outsc}/out_md_equil.pdb")
newtraj.save_xtc(f"{outsc}/out_md_equil.xtc")

  0%|          | 0/4 [00:00<?, ?it/s]

DEBUG:bioemu.sidechain_relax:creating MD setup
DEBUG:bioemu.md_utils:adding constraint force with k=1000
DEBUG:bioemu.sidechain_relax:simulation uses CUDA platform
DEBUG:bioemu.sidechain_relax:running local energy minimization


small timestep pre-equilibration:   0%|          | 0/4 [00:00<?, ?it/s]

DEBUG:bioemu.md_utils:running with init integration step of 1e-06 ps
DEBUG:bioemu.md_utils:running with init integration step of 1e-05 ps
DEBUG:bioemu.md_utils:running with init integration step of 0.0001 ps
DEBUG:bioemu.md_utils:running with init integration step of 0.001 ps
DEBUG:bioemu.md_utils:running 0.1 ns constrained MD equilibration (NVT)


NVT equilibration (0.1 ns):   0%|          | 0/100 [00:00<?, ?it/s]

...

DEBUG:bioemu.md_utils:running with init integration step of 1e-06 ps
DEBUG:bioemu.md_utils:running with init integration step of 1e-05 ps
DEBUG:bioemu.md_utils:running with init integration step of 0.0001 ps
DEBUG:bioemu.md_utils:running with init integration step of 0.001 ps
DEBUG:bioemu.md_utils:running 0.1 ns constrained MD equilibration (NVT)


NVT equilibration (0.1 ns):   0%|          | 0/100 [00:00<?, ?it/s]

DEBUG:bioemu.md_utils:running 0.4 ns constrained MD equilibration (NPT)


NPT equilibration (0.4 ns):   0%|          | 0/100 [00:00<?, ?it/s]

CPU times: user 1.01 s, sys: 449 ms, total: 1.46 s
Wall time: 1h 1min 4s


In [21]:
for i, frame in enumerate(mdtraj.load(f"{outsc}/out_md_equil.xtc", top=f"{outsc}/out_md_equil.pdb")):
    frame.save_pdb(f"{outsc}/frame_{i}.pdb")

## Non-parallel equil

In [23]:
outsc = f"{outdir}/relax_equil_nonparallel"
os.makedirs(outsc)

In [24]:
%%time

frame = run_one_md(**dict(
    frame = traj[0],
    only_energy_minimization = False,
    simtime_ns = 0.0, # default
    # simtime_ns_nvt_equil: float = 0.1, # left defaults
    # simtime_ns_npt_equil: float = 0.4, # left defaults
    outpath = outsc,
    file_prefix = f'non-parallel-equil',
))

DEBUG:bioemu.sidechain_relax:creating MD setup
DEBUG:bioemu.md_utils:adding constraint force with k=1000
DEBUG:bioemu.sidechain_relax:simulation uses CUDA platform
DEBUG:bioemu.sidechain_relax:running local energy minimization


small timestep pre-equilibration:   0%|          | 0/4 [00:00<?, ?it/s]

DEBUG:bioemu.md_utils:running with init integration step of 1e-06 ps
DEBUG:bioemu.md_utils:running with init integration step of 1e-05 ps
DEBUG:bioemu.md_utils:running with init integration step of 0.0001 ps
DEBUG:bioemu.md_utils:running with init integration step of 0.001 ps
DEBUG:bioemu.md_utils:running 0.1 ns constrained MD equilibration (NVT)


NVT equilibration (0.1 ns):   0%|          | 0/100 [00:00<?, ?it/s]

DEBUG:bioemu.md_utils:running 0.4 ns constrained MD equilibration (NPT)


NPT equilibration (0.4 ns):   0%|          | 0/100 [00:00<?, ?it/s]

CPU times: user 8min 31s, sys: 2.91 s, total: 8min 34s
Wall time: 12min 15s
